In [ ]:
#Cell 2: unzip
!unzip -q -o "plp-project-main (NEW).zip"

In [ ]:
#Cell 3 — its own cell, never chained with a ! command on the same line:
%cd /content/plp-project-main

In [ ]:
#Cell 4 — move the DB and embeddings into place:
import shutil, os
os.makedirs("data", exist_ok=True)
shutil.move("/content/plp.db", "data/plp.db")
shutil.move("/content/sentence_embeddings.npz", "data/sentence_embeddings.npz")
print("done")

In [ ]:
#Install dependencies
!pip install -q -r requirements.txt

In [ ]:
#Cell 6 — sanity-check the taxonomy loaded correctly before spending time on anything else:
import yaml
cfg = yaml.safe_load(open("config/aspects.yaml"))
print(len(cfg["aspects"]) - 1, "aspects (excl. 'other'):")
print(list(cfg["aspects"].keys()))
print("similarity_floor:", cfg["similarity_floor"])

In [ ]:
#To run GPU
targets = ["sentiment/score.py", "aspects/embed.py"]
old = '_device = "mps" if torch.backends.mps.is_available() else "cpu"'
new = ('_device = ("cuda" if torch.cuda.is_available() else '
       '"mps" if torch.backends.mps.is_available() else "cpu")')

for path in targets:
    with open(path) as f:
        content = f.read()
    if old not in content:
        print(f"pattern not found in {path} — check manually")
        continue
    with open(path, "w") as f:
        f.write(content.replace(old, new))
    print(f"patched {path}")

In [ ]:
#Cell 7 — assign every sentence to the new taxonomy:
!python -m aspects.assign

In [ ]:
# Cell 8 replacement — sentiment-score a random sample only (enough to
# stratify export_candidates), instead of the full corpus. Safe to re-run:
# it only scores pairs that don't already have a score, so bumping
# SAMPLE_SIZE and re-running just tops up what's missing.

from ingest.db import get_connection, init_db, now_iso
from sentiment.score import classify_sentences, MODEL_VERSION, _SCHEMA

SAMPLE_SIZE = 12000

init_db()
conn = get_connection()
conn.executescript(_SCHEMA)

rows = conn.execute(f"""
    SELECT sa.sentence_id, sa.aspect, s.text
    FROM sentence_aspects sa
    JOIN sentences s ON s.sentence_id = sa.sentence_id
    WHERE NOT EXISTS (
        SELECT 1 FROM aspect_sentiment ase
        WHERE ase.sentence_id = sa.sentence_id AND ase.aspect = sa.aspect
          AND ase.model_version = ?
    )
    ORDER BY RANDOM()
    LIMIT {SAMPLE_SIZE}
""", (MODEL_VERSION,)).fetchall()

texts = [r[2] for r in rows]
predictions = classify_sentences(texts)
ts = now_iso()
to_insert = [
    (sid, aspect, label, conf, MODEL_VERSION, ts)
    for (sid, aspect, _text), (label, conf) in zip(rows, predictions)
]
conn.executemany(
    """INSERT OR IGNORE INTO aspect_sentiment
       (sentence_id, aspect, label, confidence, model_version, scored_at)
       VALUES (?,?,?,?,?,?)""",
    to_insert,
)
conn.commit()

counts = conn.execute(
    "SELECT label, COUNT(*) FROM aspect_sentiment WHERE model_version=? GROUP BY label",
    (MODEL_VERSION,),
).fetchall()
conn.close()

print(f"scored {len(to_insert)} new pairs this run")
print("class counts so far:", counts)

In [ ]:
#Cell 8 — score sentiment on every assigned (sentence, aspect) pair
#!python -m sentiment.score

In [ ]:
#Cell 9 — export candidates.csv (1000 train/dev + 300 test by default
!python -m preprocess.export_candidates --train-dev 1000 --test 300

In [ ]:
#Cell 10 — export the blind test sheet (strips FinBERT's own predicted labels so you're not anchored on them while labeling):
!python -m preprocess.export_blind_test

In [ ]:
#Cell 11 — download everything back:
from google.colab import files
files.download("data/labeled/candidates.csv")
files.download("data/labeled/test_blind.csv")
files.download("data/plp.db")